<a href="https://colab.research.google.com/github/AK-Joshua-ai/Trip-advisor-sentiment-analysis-pipeline/blob/main/Trip_Advisor_Sentiment_Analysis_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INDIVIDUAL PROJECT - BIG DATA ANALYTICS (M508)

## Student Number - GH1050946
## Student Name - JOSHUA AGABA

# Business Context

Trip Advisor is a global travel guidance platform, it is also a marketplace that utilizes user generated content such as reviews and ratings to help reduce information asymmetry for those intending to travel globally. Its business model is primarily a B2B by redirecting customers to third party suppliers.

## Business Problem

Trip advisor deals with tens of thousands of customer generated reviews on a daily basis, going through each and every one of these reviews to determine customer dissatisfaction is an almost impossible task manually. Therefore developing a natural language processing pipeline that the company can deploy to carry out these audits will be very advantageous to the company and its third party suppliers.

### How solving this problem is going to benefit the company:



*   Automated processes: This will help trip advisor leverage speed to efficiently provide recommendations based on the customer generated reviews it has within its database.
*   Real-time alerts. For the case of negative reviews, these would be very useful for the third party suppliers i.e hotels whereby they can come up with corrective strategies.



### Relevant Data Collection and formulation of an NLP Task.

The data is collected from customer reviews and the star ratings assigned to a particular provider.


I will formulate this problem as a Supervised Binary Text Classification task for natural language processing.

### Main components of NLP system, interconnectedness and their functions.

The user/customer reviews and the ratings are the main components.

The reviews will be the features used to predict the label which in this case is the ratings.

#### Data Source URL: https://www.kaggle.com/datasets/andrewmvd/trip-advisor-hotel-reviews

# NLP design, implementation and analysis

## Importing Necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report

## Data Exploration & Preprocessing

Loading the dataset.

ta refers to trip advisor

In [ ]:
df_ta = pd.read_csv('/content/tripadvisor_hotel_reviews.csv')
df_ta.head(6)

,Review,Rating
0,nice hotel expensive parking got good deal sta...,4
1,ok nothing special charge diamond member hilto...,2
2,nice rooms not 4* experience hotel monaco seat...,3
3,"unique, great stay, wonderful time hotel monac...",5
4,"great stay great stay, went seahawk game aweso...",5
5,love monaco staff husband stayed hotel crazy w...,5


Data Inspection

In [ ]:
print(df_ta.info())
print(df_ta.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20491 entries, 0 to 20490
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Review  20491 non-null  object
 1   Rating  20491 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 320.3+ KB
None
                                              Review  Rating
0  nice hotel expensive parking got good deal sta...       4
1  ok nothing special charge diamond member hilto...       2
2  nice rooms not 4* experience hotel monaco seat...       3
3  unique, great stay, wonderful time hotel monac...       5
4  great stay great stay, went seahawk game aweso...       5


Data Quality Checks

In [ ]:
df_ta.isna().sum()

,0
Review,0
Rating,0


In [ ]:
df_ta.duplicated().sum()

np.int64(0)

So far the data quality checks show that there are no null values and duplicated values in this dataset.

Feature engineering

In [ ]:
df_ta = df_ta[df_ta['Review'].astype(str).str.strip() != '']

This code basically removes any rows where the review column has only blank spaces or is also empty.

In order to maintain a high signal and low noise ratio, we will get rid of the rating 3 which in this case is a neutral rating. This is because they are often mixed with good and bad making it blurry and difficult for our model to distinguish between the two.

In [ ]:
df_ta = df_ta[df_ta['Rating'] != 3].copy()

df_ta['Sentiment_label'] = df_ta['Rating'].apply(lambda x: 1 if x > 3 else 0)
df_ta = df_ta.rename(columns={'Review': 'text_ta'})
df_ta = df_ta[['text_ta', 'Sentiment_label']]



Dropping the neutral reviews ==3 will also allow us to have a binary sentiment label classification

In [ ]:
df_ta.head()

,text_ta,Sentiment_label
0,nice hotel expensive parking got good deal sta...,1
1,ok nothing special charge diamond member hilto...,0
3,"unique, great stay, wonderful time hotel monac...",1
4,"great stay great stay, went seahawk game aweso...",1
5,love monaco staff husband stayed hotel crazy w...,1


In [ ]:
df_ta['Sentiment_label'].value_counts(normalize=True)

,proportion
Sentiment_label,
1,0.824439
0,0.175561


This shows us the distribution of the positive and negative reviews.

Splitting our data into train and test before we can perform any further modifications to our data. This is mainly to prevent data leakage.

In [ ]:
train_df_ta, test_df_ta = train_test_split(df_ta, test_size=0.2, random_state=42, stratify=df_ta['Sentiment_label'])

print(f'Train set size: {len(train_df_ta)}')
print(f'Test set size: {len(test_df_ta)}')

Train set size: 14645
Test set size: 3662


Text Normalization

Now from the natural language tool kit, we shall download specific packages very necessary for text cleaning and text preprocessing.

In [ ]:
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

The quiet=True parameter is telling the tool kit to download them silently without displaying progress bars inorder to keep the notebook unclustered and neat.

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

Lemmatizer is a normalization technique that helps reduce a word down to its root form, this helps avoid looking at the same word in different states as different words.

Stop words looks at all the common words in the english language as we have specified, and helps get rid of these high frequency words since they carry no meaning and are just extra baggage for the dataset.

Defining a simple function to optimize the text cleaning, normalization and tokenization at the same time.

In [ ]:
def cleaned_text(text_ta):
  text_cleaning = re.sub(r"[^a-zA-Z\s]", "", str(text_ta).lower())
  text_cleaning = text_cleaning.split() #tokenization
  text_cleaning = [
      lemmatizer.lemmatize(word)
      for word in text_cleaning
      if word not in stop_words
  ]
  return " ".join(text_cleaning)

Using the regex, we also get rid of the punctuations and any numeric values since they are not included within the list [^a-zA-Z\s]

In [ ]:
train_df_ta['clean_text_ta'] = train_df_ta['text_ta'].apply(cleaned_text)
test_df_ta['clean_text_ta'] = test_df_ta['text_ta'].apply(cleaned_text)

In [ ]:
train_df_ta.tail(6)

,text_ta,Sentiment_label,clean_text_ta
11645,excellent hotel excellent alternative hotels c...,1,excellent hotel excellent alternative hotel cr...
19567,fabulous girls went new york great stay hotel ...,1,fabulous girl went new york great stay hotel m...
7044,does n't better husband just returned second v...,1,nt better husband returned second visit month ...
19288,"pretty good hotel small hotel nice barcelona, ...",1,pretty good hotel small hotel nice barcelona r...
20351,n't waste money stayed museum hotel nights aug...,0,nt waste money stayed museum hotel night augus...
17711,"great stayed hotel vitale night month mother, ...",1,great stayed hotel vitale night month mother t...


From this small sample can see from the cleaned text that words have been taken to their root form, and punctuations have also been taken out.

Feature Extraction - Text Representation

This is necessary for training the traditional machine learning models

TF-IDF Vectorizer

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

X_train = tfidf_vectorizer.fit_transform(train_df_ta['clean_text_ta'])
X_test = tfidf_vectorizer.transform(test_df_ta['clean_text_ta'])

y_train = train_df_ta['Sentiment_label'].values
y_test = test_df_ta['Sentiment_label'].values

This method counts word frequency within the documents in this case the different reviews, and penalizes by lowering the weights of the common words that appear everywhere so that they are of little importance in determining sentiment.

Bag of Words / Count Vectorizer

In [ ]:
# bow_vectorizer = CountVectorizer(max_features=10000, ngram_range=(1, 2))

# X_train = bow_vectorizer.fit_transform(train_df_ta['clean_text_ta'])
# X_test = bow_vectorizer.transform(test_df_ta['clean_text_ta'])

# y_train = train_df_ta['Sentiment_label'].values
# y_test = test_df_ta['Sentiment_label'].values

This method simply counts how many times a word appears in a particular review.

## Training machine learning models

Multinomial Naive Bayes, Linear Regression and Support Vector Machines

Multinomial Naive Bayes: counts how often words appear in each category and assigns a probability.

Linear Regression: statistical model used to predict a continuous numerical value rather than categorizing.

Support vector machines: draws a stategic boundary between categories.

In [ ]:
nb_model = MultinomialNB()
lr_model = LogisticRegression(C=5.0, max_iter=2000, solver='lbfgs', class_weight='balanced', random_state=42)
svm_model = LinearSVC(max_iter=2000, class_weight='balanced', random_state=42, dual=False)


In [ ]:
nb_model.fit(X_train, y_train)
nb_predictions = nb_model.predict(X_test)


lr_model.fit(X_train, y_train)
lr_predictions = lr_model.predict(X_test)


svm_model.fit(X_train, y_train)
svm_predictions = svm_model.predict(X_test)

print(f'Multinomial Naive Bayes: \n{classification_report(y_test, nb_predictions)}')
print(f'Logistic Regression: \n{classification_report(y_test, lr_predictions)}')
print(f'Support Vector Machines: \n{classification_report(y_test, svm_predictions)}')

Multinomial Naive Bayes: 
              precision    recall  f1-score   support

           0       0.94      0.62      0.75       643
           1       0.93      0.99      0.96      3019

    accuracy                           0.93      3662
   macro avg       0.93      0.81      0.85      3662
weighted avg       0.93      0.93      0.92      3662

Logistic Regression: 
              precision    recall  f1-score   support

           0       0.84      0.91      0.87       643
           1       0.98      0.96      0.97      3019

    accuracy                           0.95      3662
   macro avg       0.91      0.94      0.92      3662
weighted avg       0.96      0.95      0.95      3662

Support Vector Machines: 
              precision    recall  f1-score   support

           0       0.86      0.88      0.87       643
           1       0.98      0.97      0.97      3019

    accuracy                           0.95      3662
   macro avg       0.92      0.93      0.92      3662


According to the findings above, Logistic Regression performs best in the three machine learning model tested. This can be determined from its recall and precision scores that are better than the other two models. This means it accurately captured 91% of all negative customer reviews according to the recall and it is 84% right whenever it flags a customer review as negative according to the precision.

For a task like sentiment analysis we do not focus that much on the accuracy of the model since most real world customer reviews are not balanced.

## Fine-Tuning a Pretrained Transformer: DistilBert

Obtaining pretrained sub-word tokenizer

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

Transformer advanced text representation

In [ ]:
train_encodings = tokenizer(list(train_df_ta['text_ta']), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_df_ta['text_ta']), truncation=True, padding=True, max_length=128)

Formating the tokenized text and sentiment labels according to the hugging face library inorder to be able to train our deep learning model.

In [ ]:
ta_train_tensor_dict = {
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': list(train_df_ta['Sentiment_label'])
}

ta_test_tensor_dict = {
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels': list(test_df_ta['Sentiment_label'])
}

train_dataset_hf = Dataset.from_dict(ta_train_tensor_dict)
test_dataset_hf = Dataset.from_dict(ta_test_tensor_dict)

Creating a sequence classification head on top of the distilBert layers

In [ ]:
ta_transformer_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

num_labels set to 2 applies to the last layer of the network specifying 2 outputs which is what we need for our binary classification problem.

Training Framework

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.05,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=ta_transformer_model,
    args=training_args,
    train_dataset=train_dataset_hf,
    eval_dataset=test_dataset_hf,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer.train()
ta_transformer_outputs = trainer.predict(test_dataset_hf)
ta_transformer_predictions = np.argmax(ta_transformer_outputs.predictions, axis=1)


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,0.133232,0.098063
2,0.095738,0.126131


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Because i incorporated early stopping, it stops on the second epoch after which their is no more improvement on the model

In [ ]:
print(f'DistilBERT Classification Report: \n{classification_report(test_dataset_hf['labels'], ta_transformer_predictions, target_names=['Negative', 'Positive'])}')

DistilBERT Classification Report: 
              precision    recall  f1-score   support

    Negative       0.93      0.87      0.90       643
    Positive       0.97      0.99      0.98      3019

    accuracy                           0.97      3662
   macro avg       0.95      0.93      0.94      3662
weighted avg       0.97      0.97      0.97      3662



According to this, the transformer accurately captures 88 percent of negative reviews and is right 92 percent of the time when it flags a customer review as negative.

## Performance Measurement and Analysis

In [ ]:
ta_metrics_summary = {
    'Model Profile': [
        'DistilBert',
        'Logistic Regression'
        ],
    'Precision': [
        precision_score(y_test, ta_transformer_predictions, average='macro'),
        precision_score(y_test, lr_predictions, average='macro')
        ],
    'Recall': [
        recall_score(y_test, ta_transformer_predictions, average='macro'),
        recall_score(y_test, lr_predictions, average='macro')
    ],
    'F1 Score': [
        f1_score(y_test, ta_transformer_predictions, average='macro'),
        f1_score(y_test, lr_predictions, average='macro')
    ]
}

print(pd.DataFrame(ta_metrics_summary).to_string(index=False))

      Model Profile  Precision   Recall  F1 Score
         DistilBert   0.952548 0.929446  0.940486
Logistic Regression   0.911420 0.936069  0.923098


We use average='macro' because we want to see how well the model performs across all the reviews, not just the postive reviews since in our case the dataset is heavily imbalanced toward the positive reviews.

Table showing comparison between the two best models before and after adjustments:

<table class="matrix-table">
  <tr class="header-row">
    <th colspan="3">lr - before</th>
    <th></th>
    <th colspan="3">distilbert - before</th>
    <th></th>
  </tr>
  
  <tr>
    <td>precision</td>
    <td>recall</td>
    <td>f1</td>
    <td>Changes</td>
    <td>precision</td>
    <td>recall</td>
    <td>f1</td>
    <td>Changes</td>
  </tr>
  
  <tr>
    <td class="metric-cell">0.91142</td> <td class="metric-cell">0.936069</td> <td class="metric-cell">0.923098</td> <td>used tfidf vectorizer</td>
    <td class="metric-cell">0.966546</td>
    <td class="metric-cell">0.886981</td>
    <td class="metric-cell">0.920796</td>
    <td>learning rate = 5e-5</td>
  </tr>
  <tr>
    <td></td><td></td><td></td>
    <td></td><td></td><td></td>
    <td>weight decay = 0.01</td>
  </tr>
  <tr>
    <td></td><td></td><td></td>
    <td>max iteration = 1000</td>
    <td></td><td></td><td></td>
    <td>logging steps = 10</td>
  </tr>
  <tr>
    <td></td><td></td><td></td>
    <td></td>
    <td></td><td></td><td></td>
    <td>train & evaluation batch size = 16@</td>
  </tr>
  
  <tr class="section-break">
    <td colspan="3">after</td>
    <td></td>
    <td colspan="3">after</td>
    <td></td>
  </tr>
  
  <tr>
    <td class="metric-cell">0.912588</td>
    <td class="metric-cell">0.916615</td>
    <td class="metric-cell">0.914584</td>
    <td>used bag of words vectorizer</td>
    <td class="metric-cell">0.948814</td>
    <td class="metric-cell">0.931563</td>
    <td class="metric-cell">0.939901</td>
    <td>logging steps = 10</td>
  </tr>
  <tr>
    <td></td><td></td><td></td>
    <td></td><td></td><td></td>
    <td>train & evaluation batch size = 32@</td>
  </tr>
  <tr>
    <td></td><td></td><td></td>
    <td>max iteration = 2000</td>
    <td></td><td></td><td></td>
    <td>weight decay = 0.05</td>
  </tr>
  <tr>
    <td></td><td></td><td></td>
    <td></td>
    <td></td><td></td><td></td>
    <td>learning rate = 2e-5</td>
  </tr>
</table>

We see that the logistic regression model performed better with the TF-IDF vectorizer as opposed to bag of words.

However, according to all the performance measures that have been carried out, the distilBert model especially after further hyperparameter tuning shows the best performance for our task of sentiment analysis classification.

We can see that although precision dropped slightly, recall rose significantly. This is very important especially for cases of sentiment classification. The F1 score that balances both these metrics also rose which indicates that the model performed better after these adjustments.

# Final Discussion

### What are the strengths and limitations of distilBERT?

Strengths:
- Requires less training time and is more cost friendly to run as opposed to its parent BERT.
- Unlike traditional models, distilBert preserves sentiment of words, sequences and context making it the best candidate for a sentiment analysis task.

Limitations:
- Requires more resources in comparison with the traditional machine learning models i.e more memory and training power - GPUs
- More complicated to explain to a group of non technical managers.

### How do the pipeline results help us address customer dissatisfaction?

* Automated reponses due to high trust in the pipeline. Due to the high precision score of distilBert, trip advisor can automate customer support tickets showing customers a sense of urgence strenghtening customer loyalty.
* This pipeline will act as a trigger alarm for the third party suppliers dealing with trip advisors i.e hotels. Due the the high precision score, whenever a customer review is flagged as negative, trip advisor can automate an update to these suppliers so that they can come up with the necessary resolution tactics.

### Data driven recommendations to help reduce customer dissatisfaction?

1. Automatically link the pipeline to customer support ticket processes that are triggered by any reviews flagged as extremely negative in this case 0. This helps reduce response time from days to a few minutes.
2. Leverage a tiered pipeline, utlizing logistic regression to maximize the recall and distilBert to optimize the precision esnuring that no negative customer reviews go unnoticed.


In [ ]:
# import json

# in_path = "Trip_Advisor_Sentiment_Analysis.ipynb"
# out_path = "Trip_Advisor_Sentiment_Analysis.fixed.ipynb"

# with open(in_path, "r", encoding="utf-8") as f:
#     nb = json.load(f)

# if "metadata" in nb and "widgets" in nb["metadata"]:
#     del nb["metadata"]["widgets"]

# with open(out_path, "w", encoding="utf-8") as f:
#     json.dump(nb, f, ensure_ascii=False, indent=1)

In [ ]:
!jupyter nbconvert --to html Trip_Advisor_Sentiment_Analysis_Project.ipynb

[NbConvertApp] Converting notebook Trip_Advisor_Sentiment_Analysis_Project.ipynb to html
[NbConvertApp] Writing 387605 bytes to Trip_Advisor_Sentiment_Analysis_Project.html
